# ETS2-AI — treino na GPU 2× T4 do Kaggle (v0.4.1)

**Antes de rodar (30 segundos):**
1. Painel direito **Settings** → **Accelerator** → **GPU T4 x2**
   (precisa de conta Kaggle com telefone verificado)
2. **Run All** (Ctrl+F9)

**O que esperar (~5–10 min):** o kernel gera os dados do especialista,
detecta as 2 GPUs e treina com `tf.distribute.MirroredStrategy`.
Procure a linha `[kaggle] GPU detectada: 2x ...` no log.

**Modelo:** 13→240-240-200-200→3 = **150.203 parâmetros** (v0.4.1).
Na GPU o treino roda em float32 de propósito (a T4 roda FP64 a 1/32 da
velocidade). Para float64 (CPU/numpy): Settings → Environment variables →
`ETS2AI_F64 = 1` e desligue o acelerador.

**Resultado no final do log (isso é a prova):**
```
[kaggle] MSE treino ... | validacao ... (meta <= 0.15)
[kaggle] circuito fechado: 100% rotas, ... dock 100%
[kaggle] saida: model-weights.json + metrics.json (gpu=True, ngpus=2, dtype=float32)
```
Mesma semente e mesma matemática do repo (auto-gerado por `kaggle/build_kernel.py`).


In [ ]:
# ETS2-AI training kernel (auto-generated by ets2-ai/kaggle/build_kernel.py)
# Same code, same seed as the repo — output is validated by GitHub Actions.

# ===== ets2ai/contract.py =====
"""Shared feature/action contract for the ETS2-AI driving policy.

Every implementation (numpy training, Java on-device, TFLite, Windows bridge)
must obey EXACTLY this contract. Changing it requires regenerating weights.
"""
import json
from pathlib import Path

# ---------------------------------------------------------------------------
# Input features (12). Order is part of the wire format.
# ---------------------------------------------------------------------------
FEATURES = [
    "speed",            # m/s / 25 (truck top speed ~ 25 m/s = 90 km/h)
    "lane_offset",      # lateral offset from lane centre, m / 3.5
    "heading_error",    # heading error vs road tangent, rad / 0.6
    "curv_1",           # curvature at 8 m ahead, 1/m / 0.05
    "curv_2",           # curvature at 18 m ahead
    "curv_3",           # curvature at 30 m ahead
    "curv_4",           # curvature at 45 m ahead
    "curv_5",           # curvature at 60 m ahead
    "speed_limit",      # speed limit, m/s / 25
    "fuel",             # fuel fraction 0..1
    "fatigue",          # fatigue 0..1 (1 = must sleep immediately)
    "job_dist",         # remaining job distance, km / 100
    "radar_dist",       # distance to the next speed camera, m / 500
]
N_IN = len(FEATURES)

# ---------------------------------------------------------------------------
# Outputs (3). Order is part of the wire format.
# ---------------------------------------------------------------------------
ACTIONS = ["steer", "throttle", "brake"]
N_OUT = len(ACTIONS)

# MLP architecture (part of the wire format).
HIDDEN = [240, 240, 200, 200]   # 150,203 parameters (~150k, user spec v0.4.1)
SEED = 20260930  # deterministic training across runs/CI

# Reporting target demanded by the project brief.
LOSS_TARGET = 0.150


def clamp_action(steer, throttle, brake):
    """Actuator clamp — part of the wire contract (mirrored in Java/bridge).
    The raw MLP may extrapolate outside the training distribution; commands
    sent to the game (or the demo sim) are always clamped here."""
    return (max(-1.0, min(1.0, steer)),
            max(0.0, min(1.0, throttle)),
            max(0.0, min(1.0, brake)))


def model_meta(epochs=None, samples=None, loss=None, dtype=None):
    meta = {
        "features": FEATURES,
        "actions": ACTIONS,
        "hidden": HIDDEN,
        "seed": SEED,
        "loss_target": LOSS_TARGET,
        "epochs": epochs,
        "samples": samples,
        "final_loss": loss,
    }
    if dtype is not None:  # float32 (deploy) nao registra: mantem o JSON igual
        meta["dtype"] = dtype
    return meta


def save_weights(path, layer_weights, meta):
    """layer_weights: list of (W, b) numpy arrays; saved as plain JSON."""
    payload = {
        "meta": meta,
        "layers": [
            {"w": w.tolist(), "b": b.tolist()} for (w, b) in layer_weights
        ],
    }
    Path(path).write_text(json.dumps(payload), encoding="utf-8")


def load_weights(path):
    import numpy as np
    payload = json.loads(Path(path).read_text(encoding="utf-8"))
    layers = [
        (np.array(l["w"], dtype=np.float32), np.array(l["b"], dtype=np.float32))
        for l in payload["layers"]
    ]
    return layers, payload["meta"]


def save_device_format(path, layers, meta):
    """Plain-text weights for the Java app (no JSON parser on device).

    Format (CSV, one float per value, '.' decimal separator):
        ETS2AI,v1,<nin>,<nout>,<h1>,<h2>
        per layer: W rows (fan_in lines of fan_out floats), then bias line.
    The values are bit-identical to model-weights.json.
    """
    lines = [f"ETS2AI,v1,{N_IN},{N_OUT},{','.join(str(h) for h in HIDDEN)}"]
    for (w, b) in layers:
        for row in w:
            lines.append(",".join(repr(float(v)) for v in row))
        lines.append(",".join(repr(float(v)) for v in b))
    Path(path).write_text("\n".join(lines) + "\n", encoding="utf-8")


def load_device_format(path):
    """Reference loader (used by tests to prove txt == json)."""
    import numpy as np
    lines = Path(path).read_text(encoding="utf-8").strip().splitlines()
    header = lines[0].split(",")
    assert header[0] == "ETS2AI" and header[1] == "v1"
    sizes = [int(header[2])] + [int(h) for h in header[4:]] + [int(header[3])]
    layers, pos = [], 1
    for i in range(len(sizes) - 1):
        fan_in, fan_out = sizes[i], sizes[i + 1]
        w = np.zeros((fan_in, fan_out), dtype=np.float32)
        for r in range(fan_in):
            w[r] = [float(v) for v in lines[pos].split(",")]
            pos += 1
        b = np.array([float(v) for v in lines[pos].split(",")], dtype=np.float32)
        pos += 1
        layers.append((w, b))
    assert pos == len(lines)
    return layers
# ===== ets2ai/sim.py =====
"""ETS2-AI world model: road, truck physics, expert driver, career rules.

Deterministic, dependency-light, ported 1:1 to Java (SimWorld.java) and to the
Windows bridge demo so that the phone, the PC and the trainer all agree on
physics and the expert never disagrees with itself.
"""
import math
import numpy as np

DT = 0.1                 # simulation step (s)
WHEELBASE = 4.0          # m (kinematic bicycle)
MAX_STEER = 0.6          # rad at the wheels, scaled by |steer| command
MAX_ACCEL = 1.8          # m/s^2 at full throttle
MAX_BRAKE = 3.2          # m/s^2 at full brake
DRAG = 0.030             # rolling/aero deceleration factor per (m/s)
LAT_ACCEL_COMFORT = 2.4  # m/s^2 the expert accepts in curves
LAT_ACCEL_ROLLOVER = 3.6  # m/s^2 where a truck starts to roll over
ROLLOVER_MARGIN = 0.82    # the expert stays at 82% of the rollover limit
V_MAX = 36.0              # m/s (~130 km/h): speeding policy
RADAR_SLOW = 0.95         # fraction of the limit kept AT the camera
RADAR_BRAKE = 2.6         # m/s^2 deceleration budget before a camera
LANE_HALF = 2.0          # |offset| beyond this = off-lane
ROAD_HALF = 4.6          # |offset| beyond this = off-road (crash barrier)

FUEL_START = 1.0
FUEL_BURN_IDLE = 0.000012      # per second
FUEL_BURN_FULL = 0.000160      # per second at full throttle
FATIGUE_START = 0.0
FATIGUE_RATE = 1.0 / (11 * 3600)   # ~11 h of driving to full fatigue
REFUEL_BELOW = 0.14
SLEEP_ABOVE = 0.82
FUEL_PRICE = 1.45              # EUR / L  (tank modelled as fraction)
TANK_LITERS = 600
SLEEP_HOURS = 9
SPEED_LIMIT_MPS = 25.0         # 90 km/h truck limit

LOOKAHEAD = (8.0, 18.0, 40.0, 90.0, 170.0)
PURE_PURSUIT_DIST = 12.0


# ---------------------------------------------------------------------------
# Road: piecewise-constant-curvature ribbon, seeded deterministically.
# ---------------------------------------------------------------------------
class Road:
    def __init__(self, seg_lengths, seg_curvs):
        self.seg_lengths = [float(v) for v in seg_lengths]
        self.seg_curvs = [float(v) for v in seg_curvs]
        self.cum = [0.0]
        for L in self.seg_lengths:
            self.cum.append(self.cum[-1] + L)
        self.length = self.cum[-1]
        self.radars = []
        # integrate centreline
        pts = [(0.0, 0.0)]
        hdg = 0.0
        step = 2.0
        s = 0.0
        self.samples = [pts[0]]
        self.sample_s = [0.0]
        while s < self.length:
            s = min(s + step, self.length)
            hdg += self.curvature_at(s) * step
            x, y = self.samples[-1]
            self.samples.append((x + step * math.cos(hdg),
                                 y + step * math.sin(hdg)))
            self.sample_s.append(s)

    @staticmethod
    def random(seed, n_seg=40, min_len=60.0, max_len=420.0, max_curv=0.012):
        rng = np.random.default_rng(seed)
        lens = rng.uniform(min_len, max_len, n_seg)
        curvs = []
        for _ in range(n_seg):
            r = rng.uniform()
            if r < 0.35:                    # straight
                curvs.append(0.0)
            else:
                curvs.append(float(rng.uniform(-max_curv, max_curv)))
        road = Road(lens, curvs)
        # speed cameras along the route (the "radars")
        rng2 = np.random.default_rng(seed + 777)
        radars = []
        pos = 700.0
        while pos < road.length - 150.0:
            radars.append(pos)
            pos += float(rng2.uniform(1500.0, 2800.0))
        road.radars = radars
        return road

    def upcoming_curves(self, s, max_dist=260.0):
        """[(distance_to_start, |kappa|)] of curves ahead, in road order."""
        out = []
        for i, k in enumerate(self.seg_curvs):
            seg_start, seg_end = self.cum[i], self.cum[i + 1]
            if seg_end <= s:
                continue
            d = max(0.0, seg_start - s)
            if d > max_dist:
                break
            if abs(k) > 1e-6:
                out.append((d, abs(k)))
        return out

    def radar_dist_ahead(self, s):
        """Metres to the next camera ahead of s (500 = none ahead)."""
        best = 500.0
        for r in self.radars:
            d = r - s
            if 0.0 < d < best:
                best = d
        return best

    def curvature_at(self, s):
        s = max(0.0, min(s, self.length - 1e-6))
        for i, L in enumerate(self.seg_lengths):
            if s <= self.cum[i + 1] + 1e-9:
                return self.seg_curvs[i]
        return 0.0

    def project(self, x, y, hint=None):
        """Closest centreline sample -> (s, offset, tangent_heading).

        `hint`: last sample index; searches a local window first (the truck
        moves a few metres per step). Falls back to a full scan. Same
        behaviour in the Java port.
        """
        samples = self.samples
        n = len(samples)
        if hint is not None:
            lo, hi = max(0, hint - 40), min(n, hint + 40)
            best_i, best_d2 = None, float("inf")
            for i in range(lo, hi):
                px, py = samples[i]
                d2 = (px - x) ** 2 + (py - y) ** 2
                if d2 < best_d2:
                    best_i, best_d2 = i, d2
            if best_i is not None and best_d2 < (30.0 ** 2):
                return self._project_at(best_i, x, y)
        best_i, best_d2 = 0, float("inf")
        for i, (px, py) in enumerate(samples):
            d2 = (px - x) ** 2 + (py - y) ** 2
            if d2 < best_d2:
                best_i, best_d2 = i, d2
        return self._project_at(best_i, x, y)

    def _project_at(self, best_i, x, y):
        s = self.sample_s[best_i]
        i0 = max(0, best_i - 2)
        i1 = min(len(self.samples) - 1, best_i + 2)
        tx = self.samples[i1][0] - self.samples[i0][0]
        ty = self.samples[i1][1] - self.samples[i0][1]
        tn = math.hypot(tx, ty) or 1.0
        tx, ty = tx / tn, ty / tn
        dx, dy = x - self.samples[best_i][0], y - self.samples[best_i][1]
        offset = dx * -ty + dy * tx
        return s, offset, math.atan2(ty, tx)


# ---------------------------------------------------------------------------
# Truck state + physics step (identical in Java/Python).
# ---------------------------------------------------------------------------
class Truck:
    def __init__(self, road, s=0.0, offset=0.0, speed=15.0, heading=None):
        self.s = s
        self._hint = 0
        x, y, th = road_point(road, s)
        # start on the lane line given by offset (positive = right of centre)
        self.x = x - offset * math.sin(th)
        self.y = y + offset * math.cos(th)
        self.s, self.offset, self.tangent = self.project(road)
        self.heading = self.tangent if heading is None else heading
        self.speed = speed
        self.steer_pos = 0.0  # smoothed actual steering, for features
        self.fuel = FUEL_START
        self.fatigue = FATIGUE_START
        self.money = 0.0
        self.driven_km = 0.0

    def project(self, road):
        """Fast re-projection using a moving window (kept in Java too)."""
        samples = road.samples
        n = len(samples)
        best_i, best_d2 = self._hint, float("inf")
        for i in range(max(0, self._hint - 40), min(n, self._hint + 40)):
            px, py = samples[i]
            d2 = (px - self.x) ** 2 + (py - self.y) ** 2
            if d2 < best_d2:
                best_i, best_d2 = i, d2
        if best_d2 > 30.0 ** 2:
            return road.project(self.x, self.y)   # full scan fallback
        self._hint = best_i
        return road._project_at(best_i, self.x, self.y)

    def step(self, road, steer, throttle, brake, dt=DT):
        steer = max(-1.0, min(1.0, steer))
        throttle = max(0.0, min(1.0, throttle))
        brake = max(0.0, min(1.0, brake))
        # steering actuator: rack moves toward command
        self.steer_pos += max(-0.15, min(0.15, steer - self.steer_pos))
        wheel = self.steer_pos * MAX_STEER
        self.heading += (self.speed / WHEELBASE) * math.tan(wheel) * dt
        self.x += self.speed * math.cos(self.heading) * dt
        self.y += self.speed * math.sin(self.heading) * dt
        acc = throttle * MAX_ACCEL - brake * MAX_BRAKE - DRAG * self.speed
        self.speed = max(0.0, self.speed + acc * dt)
        self.driven_km += self.speed * dt / 1000.0
        self.fuel = max(0.0, self.fuel - dt *
                        (FUEL_BURN_IDLE + throttle * (FUEL_BURN_FULL - FUEL_BURN_IDLE)))
        self.fatigue = min(1.0, self.fatigue + dt * FATIGUE_RATE)
        self.s, self.offset, self.tangent = self.project(road)


def road_point(road, s):
    """(x, y, heading) of centreline near arclength s (nearest sample)."""
    i = min(range(len(road.sample_s)), key=lambda k: abs(road.sample_s[k] - s))
    th = heading_at(road, i)
    x, y = road.samples[i]
    return x, y, th


def heading_at(road, i):
    i0, i1 = max(0, i - 1), min(len(road.samples) - 1, i + 1)
    return math.atan2(road.samples[i1][1] - road.samples[i0][1],
                      road.samples[i1][0] - road.samples[i0][0])


# ---------------------------------------------------------------------------
# Features / actions (the contract in numbers).
# ---------------------------------------------------------------------------
def features(road, truck, job_left_km):
    s, offset, tangent = truck.project(road)
    hdg_err = wrap_angle(truck.heading - tangent)
    curvs = [road.curvature_at(s + d) / 0.05 for d in LOOKAHEAD]
    return [
        truck.speed / 25.0,
        offset / 3.5,
        hdg_err / 0.6,
        curvs[0], curvs[1], curvs[2], curvs[3], curvs[4],
        SPEED_LIMIT_MPS / 25.0,
        truck.fuel,
        truck.fatigue,
        min(job_left_km, 20.0) / 20.0,
        road.radar_dist_ahead(s) / 500.0,
    ]


def wrap_angle(a):
    while a > math.pi:
        a -= 2 * math.pi
    while a < -math.pi:
        a += 2 * math.pi
    return a


# ---------------------------------------------------------------------------
# Rollover governor (ESC-like): caps speed so v^2 * |kappa| stays below the
# rollover limit. The net proposes; the governor guarantees the physics.
# Mirrored in SimWorld.java and in the bridge.
# ---------------------------------------------------------------------------
GOVERNOR_LIMIT = 0.92 * LAT_ACCEL_ROLLOVER


def governor(road, truck, cmd):
    """ESC: full brake if the current speed cannot be shed before the START
    of any curve ahead (brake planned to real curve-start distance)."""
    worst = None
    for d, k in road.upcoming_curves(truck.s):
        v_curve = math.sqrt(GOVERNOR_LIMIT / k)
        v_allow = math.sqrt(v_curve ** 2 + 2.0 * MAX_BRAKE * max(0.0, d - 8.0))
        if worst is None or v_allow < worst:
            worst = v_allow
    if worst is not None and truck.speed > worst:
        return (cmd[0], 0.0, 1.0)  # full brake: safety over comfort
    # dock governor: never carry speed into the loading dock (full distance)
    d_dock = (road.length - 6.0) - truck.s
    if 0.0 < d_dock < 600.0:
        v_allow = math.sqrt(2.0 * MAX_BRAKE * max(0.0, d_dock - 2.0))
        if truck.speed > v_allow + 0.3:
            return (cmd[0], 0.0, 1.0)
        if d_dock <= 1.5:
            return (cmd[0], 0.0, 1.0)               # hold at the line
        if d_dock < 170.0 and truck.speed < 0.6:
            return (cmd[0], max(cmd[1], 0.35), 0.0)  # creep: never stall short
    return cmd


# ---------------------------------------------------------------------------
# Expert driver: pure-pursuit steering + predictive speed control.
# ---------------------------------------------------------------------------
def expert(road, truck, job_left_km, noise=0.0, rng=None):
    s, offset, tangent = truck.project(road)
    tx, ty = math.cos(tangent), math.sin(tangent)
    d = PURE_PURSUIT_DIST
    gs = s + d
    gx = road_point(road, gs)[0] if gs <= road.length else None
    # target point on centreline ahead
    i = min(range(len(road.sample_s)), key=lambda k: abs(road.sample_s[k] - gs))
    px, py = road.samples[i]
    hdg_err = wrap_angle(truck.heading - tangent)
    lat_err = offset
    # pure pursuit: steer toward eliminating combined error
    alpha = math.atan2(-lat_err, d) - hdg_err
    steer = 2.0 * math.sin(alpha) / max(d, 1e-3) * 12.0
    steer = max(-1.0, min(1.0, steer))
    # speed target: as fast as possible WITHOUT ROLLING OVER — planning the
    # brake against the real distance to each curve START (not sample points).
    v_target = V_MAX
    a_lat = LAT_ACCEL_ROLLOVER * ROLLOVER_MARGIN
    for d, k in road.upcoming_curves(s):
        v_curve = min(V_MAX, math.sqrt(a_lat / k))
        v_allow = math.sqrt(v_curve ** 2 + 2.0 * RADAR_BRAKE * max(0.0, d - 12.0))
        v_target = min(v_target, v_allow)
    # dock: precise stop at the end of the route — continuous profile that
    # begins easing off hundreds of metres out (a fixed window cannot shed
    # 130 km/h in 130 m; physics decides the distance, not a constant).
    d_dock = (road.length - 6.0) - s
    if d_dock < 600.0:
        v_target = min(v_target, math.sqrt(2.0 * 1.6 * max(0.0, d_dock)))
        if v_target < 0.4:
            v_target = 0.0
    # radar: only slow down within braking distance of the camera
    d_radar = road.radar_dist_ahead(s)
    v_radar = SPEED_LIMIT_MPS * RADAR_SLOW
    brake_dist = max(0.0, truck.speed ** 2 - v_radar ** 2) / (2.0 * RADAR_BRAKE) + 25.0
    if d_radar < brake_dist:
        v_target = min(v_target, v_radar)
    # fatigue makes the expert ease off slightly (safe behaviour to imitate)
    v_target *= (1.0 - 0.08 * truck.fatigue)
    dv = v_target - truck.speed
    throttle = max(0.0, min(1.0, dv / 1.8))
    brake = max(0.0, min(1.0, -dv / 2.6))
    if noise and rng is not None:
        steer = max(-1.0, min(1.0, steer + rng.normal(0, noise)))
        throttle = max(0.0, min(1.0, throttle + rng.normal(0, noise * 0.5)))
        brake = max(0.0, min(1.0, brake + rng.normal(0, noise * 0.5)))
    return steer, throttle, brake


# ---------------------------------------------------------------------------
# Closed-loop episode runner (used for data generation AND evaluation).
# ---------------------------------------------------------------------------
def run_episode(road, policy, seed=0, max_steps=7000, noise=0.0,
                record=False, start_jitter=True):
    """policy(road, truck, job_left_km) -> (steer, throttle, brake)."""
    rng = np.random.default_rng(seed)
    if start_jitter:
        truck = Truck(road, s=5.0,
                      offset=float(rng.uniform(-2.0, 2.0)),
                      speed=float(rng.uniform(8.0, 22.0)))
        truck.heading += float(rng.uniform(-0.25, 0.25))
    else:
        truck = Truck(road, s=5.0, offset=0.0, speed=15.0)
    job_km = road.length / 1000.0
    feats, acts = [], []
    in_lane = off_road = 0
    stopped = False
    steps = 0
    radar_pass = []          # speeds AT each camera
    max_lat = 0.0            # max lateral acceleration (rollover watch)
    off_sum = 0.0
    docked = False
    stop_error_m = None
    while steps < max_steps:
        s, offset, _ = truck.project(road)
        d_dock = (road.length - 6.0) - s
        if d_dock <= 4.0 and truck.speed < 0.6:      # parked AT the line
            break
        if s >= road.length - 1.0:
            break
        job_left = max(0.0, (road.length - s) / 1000.0)
        steer, throttle, brake = policy(road, truck, job_left, rng)
        steer, throttle, brake = governor(road, truck, (steer, throttle, brake))
        if record:
            feats.append(features(road, truck, job_left))
            acts.append([steer, throttle, brake])
        prev_s = s
        truck.step(road, steer, throttle, brake)
        steps += 1
        d_dock_now = (road.length - 6.0) - truck.s
        if d_dock_now <= 4.0 and truck.speed < 0.6:
            docked = True
            stop_error_m = d_dock_now
            break
        if truck.s >= road.length - 1.0:
            stop_error_m = (road.length - 6.0) - truck.s   # overshoot
            break
        for r in road.radars:                     # crossed a camera?
            if prev_s < r <= truck.s:
                radar_pass.append(truck.speed)
        max_lat = max(max_lat, truck.speed ** 2 * abs(road.curvature_at(truck.s)))
        off_sum += abs(truck.offset)
        if abs(offset) < LANE_HALF:
            in_lane += 1
        if (abs(offset) > ROAD_HALF
                or (truck.speed < 0.3 and steps > 60
                    and (road.length - truck.s) > 170.0)):
            off_road += 1
            break
        # career rules (identical in the app): refuel / sleep
        if truck.fuel < REFUEL_BELOW:
            truck.fuel = 1.0
            truck.money -= FUEL_PRICE * TANK_LITERS * (1.0 - REFUEL_BELOW)
            stopped = True
        if truck.fatigue > SLEEP_ABOVE:
            truck.fatigue = 0.0
            truck.money -= 220.0  # hotel
            stopped = True
    km = truck.driven_km
    metrics = {
        "steps": steps,
        "in_lane_pct": in_lane / max(1, steps),
        "finished": docked and steps < max_steps,
        "docked": docked,
        "stop_error_m": stop_error_m,
        "km": km,
        "money": truck.money,
        "stopped_for_rules": stopped,
        "avg_speed": km / max(0.1, steps * DT) * 1000,
        "radar_compliance": (sum(1 for v in radar_pass if v <= SPEED_LIMIT_MPS * 1.05)
                             / max(1, len(radar_pass))),
        "radar_passes": len(radar_pass),
        "max_lat_accel": max_lat,
        "mean_abs_offset": off_sum / max(1, steps),
        "top_speed": max([0.0] + [a for a in [truck.speed]]) if steps else 0.0,
    }
    if record:
        return np.array(feats, dtype=np.float32), np.array(acts, dtype=np.float32), metrics
    return metrics
import types as _types
sim = _types.SimpleNamespace(
    features=features, Road=Road, Truck=Truck, expert=expert,
    run_episode=run_episode, wrap_angle=wrap_angle,
    LOOKAHEAD=LOOKAHEAD, SPEED_LIMIT_MPS=SPEED_LIMIT_MPS,
    REFUEL_BELOW=REFUEL_BELOW, SLEEP_ABOVE=SLEEP_ABOVE,
    ROAD_HALF=ROAD_HALF, LANE_HALF=LANE_HALF, DT=DT,
)

# ===== ets2ai/model.py =====
"""ETS2-AI core: numpy-only MLP + training + export.

Deliberately dependency-free (numpy only) so the exact same numbers can be
reproduced on the CI runner and re-implemented bit-for-bit in Java on the
phone and in TFLite. Deterministic under SEED.
"""
import json
import math
from pathlib import Path
import numpy as np



def init_layers(rng, dtype=np.float32):
    """He initialisation for tanh layers; returns [(W,b), ...]."""
    sizes = [N_IN] + list(HIDDEN) + [N_OUT]
    layers = []
    for i in range(len(sizes) - 1):
        fan_in, fan_out = sizes[i], sizes[i + 1]
        # cast DEPOIS da multiplicacao: com NEP 50 (numpy>=2), float32 * np.float64
        # promoveria para float64 — era o comportamento antigo por acidente.
        w = (rng.standard_normal((fan_in, fan_out)) * np.sqrt(1.0 / fan_in)).astype(dtype)
        b = np.zeros(fan_out, dtype=dtype)
        layers.append((w, b))
    return layers


def forward(x, layers, keep=None):
    """Forward pass. x: (N, N_IN) or (N_IN,). tanh hidden, linear output.

    Precision follows the weights (float32 deploy / float64 experiment).
    `keep` (optional) receives intermediate activations for backprop.
    Returns activations of the output layer.
    """
    a = np.atleast_2d(np.asarray(x, dtype=layers[0][0].dtype))
    last = len(layers) - 1
    for i, (w, b) in enumerate(layers):
        z = a @ w + b
        a = z if i == last else np.tanh(z)
        if keep is not None:
            keep.append(a)
    return a


def mse(pred, target):
    """MSE na precisao nativa dos inputs (float32 fica float32; float64
    conserva a faixa ~1e-308 — usado pelo modo --dtype float64)."""
    p = np.asarray(pred)
    t = np.asarray(target)
    if p.dtype != t.dtype:
        p, t = p.astype(np.float64), t.astype(np.float64)
    return float(np.mean((p - t) ** 2))


class Adam:
    def __init__(self, layers, lr=2e-3, b1=0.9, b2=0.999, eps=1e-8):
        self.lr, self.b1, self.b2, self.eps = lr, b1, b2, eps
        self.m = [[np.zeros_like(p) for p in (w, b)] for (w, b) in layers]
        self.v = [[np.zeros_like(p) for p in (w, b)] for (w, b) in layers]
        self.t = 0

    def step(self, layers, grads):
        self.t += 1
        b1c = 1 - self.b1 ** self.t
        b2c = 1 - self.b2 ** self.t
        for li, ((w, b), (gw, gb)) in enumerate(zip(layers, grads)):
            for pi, (p, g) in enumerate(((w, gw), (b, gb))):
                self.m[li][pi] = self.b1 * self.m[li][pi] + (1 - self.b1) * g
                self.v[li][pi] = self.b2 * self.v[li][pi] + (1 - self.b2) * g * g
                m_hat = self.m[li][pi] / b1c
                v_hat = self.v[li][pi] / b2c
                p -= self.lr * m_hat / (np.sqrt(v_hat) + self.eps)


def train(x, y, epochs=400, batch=512, lr=2e-3, seed=SEED, verbose=True,
          x_val=None, y_val=None, start_layers=None, dtype=np.float32,
          schedule="cosine"):
    """Train the MLP on (x, y) with Adam + MSE. Returns (layers, history).

    start_layers: optional [(W,b), ...] to continue training from existing
    weights (DAgger finetuning) instead of initialising from scratch.
    dtype: np.float32 (deploy rapido) ou np.float64 (experimento de precisao).
    schedule: "cosine" decai lr de `lr` ate 5% de `lr` ao longo das epocas
    (deterministico, ajuda a fechar a loss); None mantem lr constante.
    """
    rng = np.random.default_rng(seed)
    layers = start_layers if start_layers is not None else init_layers(rng, dtype)
    opt = Adam(layers, lr=lr)
    x = np.asarray(x, dtype=dtype)
    y = np.asarray(y, dtype=dtype)
    if x_val is not None:
        x_val = np.asarray(x_val, dtype=dtype)
        y_val = np.asarray(y_val, dtype=dtype)
    n = len(x)
    history = []
    best = (np.inf, None)
    for epoch in range(epochs):
        if schedule == "cosine":
            opt.lr = lr * (0.05 + 0.95 * 0.5 * (1.0 + math.cos(math.pi * epoch / epochs)))
        idx = rng.permutation(n)
        for s in range(0, n, batch):
            sel = idx[s:s + batch]
            acts = []
            out = forward(x[sel], layers, keep=acts)
            err = out - y[sel]                       # (B, N_OUT)
            grads = [None] * len(layers)
            delta = (2.0 / (len(sel) * N_OUT)) * err  # dMSE/dz_out (linear)
            for li in range(len(layers) - 1, -1, -1):
                a_prev = x[sel] if li == 0 else acts[li - 1]
                grads[li] = (a_prev.T @ delta, delta.sum(axis=0))
                if li > 0:
                    delta = (delta @ layers[li][0].T) * (1 - acts[li - 1] ** 2)
            opt.step(layers, grads)
        tr = mse(forward(x, layers), y)
        va = mse(forward(x_val, layers), y_val) if x_val is not None else None
        history.append((tr, va))
        if va is not None and va < best[0]:
            best = (va, [(w.copy(), b.copy()) for (w, b) in layers])
        if verbose and (epoch % 40 == 0 or epoch == epochs - 1):
            msg = f"epoch {epoch:4d}  train_mse {tr:.5f}"
            if va is not None:
                msg += f"  val_mse {va:.5f}"
            print(msg, flush=True)
    if best[1] is not None:  # restore best-validation weights
        layers = best[1]
    return layers, history
# ===== ets2ai/data.py =====
"""Dataset generation: expert demonstrations on random roads.

The expert is a pure-pursuit + predictive-speed controller with small
human-like action noise, so the learned policy imitates a competent (not
superhuman) trucker. Train/val roads are disjoint.
"""
import numpy as np


N_TRAIN_ROADS = 26
N_VAL_ROADS = 7
NOISE = 0.020          # small human imperfection (tight GPS line)
ROAD_SEED = 31337


def _oversample_dock(feats, acts, reps=4):
    """Duplicate approach/dock rows (last 150 m) so the policy learns the
    precise stop: they are a tiny fraction of each episode."""
    tail = feats[:, 11] * 20.0 * 1000.0 < 150.0     # job_left < 150 m
    return (np.concatenate([feats] + [feats[tail]] * (reps - 1)),
            np.concatenate([acts] + [acts[tail]] * (reps - 1)))


def generate():
    x_train, y_train, x_val, y_val = [], [], [], []
    for k in range(N_TRAIN_ROADS + N_VAL_ROADS):
        road = Road.random(ROAD_SEED + k)
        feats, acts, _ = run_episode(
            road,
            lambda r, t, j, rng: expert(r, t, j, noise=NOISE, rng=rng),
            seed=1000 + k, record=True)
        feats, acts = _oversample_dock(feats, acts)
        if k < N_TRAIN_ROADS:
            x_train.append(feats); y_train.append(acts)
        else:
            x_val.append(feats); y_val.append(acts)
    return (np.concatenate(x_train), np.concatenate(y_train),
            np.concatenate(x_val), np.concatenate(y_val))


# ===== ets2ai/train.py =====
"""Train the ETS2-AI driving policy and export weights + metrics.

Usage:  python -m ets2ai.train [--epochs N] [--out DIR]

Reports MSE (the project target is <= 0.150) and closed-loop driving quality
on held-out roads. Deterministic: same seed -> same weights everywhere.
"""
import argparse
import json
from pathlib import Path
import numpy as np



def policy_from_layers(layers):
    def policy(road, truck, job_left_km, rng=None):
        f = np.array(sim.features(road, truck, job_left_km), dtype=np.float32)
        out = forward(f, layers)[0]
        return float(out[0]), float(out[1]), float(out[2])
    return policy


def closed_loop_eval(layers, n_roads=N_VAL_ROADS, seed=777):
    policy = policy_from_layers(layers)
    rows = []
    for k in range(n_roads):
        road = Road.random(ROAD_SEED + 100 + k)   # roads never seen in training
        rows.append(run_episode(road, policy, seed=seed + k))
    n_pass = sum(r["radar_passes"] for r in rows)
    agg = {
        "roads": n_roads,
        "finish_rate": sum(r["finished"] for r in rows) / n_roads,
        "in_lane_pct": sum(r["in_lane_pct"] for r in rows) / n_roads,
        "avg_speed_kmh": sum(r["avg_speed"] for r in rows) / n_roads * 3.6,
        "km_total": sum(r["km"] for r in rows),
        "radar_compliance": (sum(r["radar_passes"] * r["radar_compliance"] for r in rows)
                             / max(1, n_pass)),
        "radar_passes": n_pass,
        "max_lat_accel": max(r["max_lat_accel"] for r in rows),
        "gps_mean_abs_offset_m": sum(r["mean_abs_offset"] for r in rows) / n_roads,
        "dock_rate": sum(1 for r in rows if r["docked"]) / n_roads,
        "mean_stop_error_m": (sum(r["stop_error_m"] for r in rows if r["stop_error_m"] is not None)
                              / max(1, sum(1 for r in rows if r["stop_error_m"] is not None))),
        "top_speed_kmh": max(r["avg_speed"] for r in rows) * 3.6,
    }
    return agg, rows


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--epochs", type=int, default=350)
    ap.add_argument("--out", default=str(Path(__file__).resolve().parent.parent / "artifacts"))
    ap.add_argument("--dtype", choices=["float32", "float64"], default="float64",
                    help="float64 = canônico (desde v0.4.1, exigência do usuário); "
                         "float32 = modo rápido/deploy-check")
    args = ap.parse_args()

    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)
    dtype = {"float32": np.float32, "float64": np.float64}[args.dtype]

    print("[1/4] gerando dados do especialista (estradas aleatorias)...")
    x_train, y_train, x_val, y_val = generate()
    print(f"      treino {x_train.shape}  val {x_val.shape}")

    n_params = (N_IN * HIDDEN[0] + HIDDEN[0]
                + sum(HIDDEN[i] * HIDDEN[i + 1] + HIDDEN[i + 1] for i in range(len(HIDDEN) - 1))
                + HIDDEN[-1] * N_OUT + N_OUT)
    print(f"[2/4] treinando MLP {N_IN}-{'-'.join(str(h) for h in HIDDEN)}-{N_OUT} "
          f"({n_params:,} params, tanh, Adam+cosine, {args.dtype})...")
    layers, hist = train(x_train, y_train, epochs=args.epochs,
                         x_val=x_val, y_val=y_val, dtype=dtype)

    loss_tr = mse(forward(x_train, layers), y_train)
    loss_va = mse(forward(x_val, layers), y_val)
    mae_va = float(np.mean(np.abs(forward(x_val, layers) - y_val)))
    print(f"      MSE treino {loss_tr:.5f} | MSE validacao {loss_va:.5f} "
          f"(meta <= {LOSS_TARGET}) | MAE val {mae_va:.5f}")

    print("[3/4] avaliacao em circuito fechado (estradas ineditas)...")
    agg, rows = closed_loop_eval(layers)
    print(f"      conclusao {agg['finish_rate']*100:.0f}%  "
          f"na faixa {agg['in_lane_pct']*100:.0f}%  "
          f"vel media {agg['avg_speed_kmh']:.0f} km/h")

    print("[4/4] exportando pesos...")
    meta = model_meta(epochs=args.epochs, samples=int(len(x_train) + len(x_val)),
                      loss=loss_va,
                      dtype=args.dtype if args.dtype != "float32" else None)
    save_weights(out / "model-weights.json", layers, meta)
    save_device_format(out / "model-weights.txt", layers, meta)
    (out / "metrics.json").write_text(json.dumps({
        "meta": meta,
        "mse_train": loss_tr,
        "mse_val": loss_va,
        "mae_val": mae_va,
        "loss_target": LOSS_TARGET,
        "target_met": bool(loss_va <= LOSS_TARGET),
        "closed_loop": agg,
        "closed_loop_rows": rows,
    }, indent=2), encoding="utf-8")
    print(f"      -> {out/'model-weights.json'}")
    print(f"      -> {out/'metrics.json'}")
    ok = loss_va <= LOSS_TARGET
    print("META ATINGIDA" if ok else "meta NAO atingida (ver --epochs)")
    return 0 if ok else 1




# ---------------------------------------------------------------------------
# Kaggle runner footer (appended by build_kernel.py — do not edit here)
# ---------------------------------------------------------------------------
import json as _json
import os as _os

_OUT = _os.environ.get("KAGGLE_WORKING_DIR", ".")
_DTYPE = np.float64 if _os.environ.get("ETS2AI_F64") == "1" else np.float32
_n_params = (N_IN * HIDDEN[0] + HIDDEN[0]
             + sum(HIDDEN[i] * HIDDEN[i + 1] + HIDDEN[i + 1] for i in range(len(HIDDEN) - 1))
             + HIDDEN[-1] * N_OUT + N_OUT)
print("[kaggle] gerando dados do especialista (estradas aleatorias)...")
_xtr, _ytr, _xva, _yva = generate()
_xtr = np.asarray(_xtr, dtype=_DTYPE); _ytr = np.asarray(_ytr, dtype=_DTYPE)
_xva = np.asarray(_xva, dtype=_DTYPE); _yva = np.asarray(_yva, dtype=_DTYPE)
print(f"[kaggle] treino {_xtr.shape} | val {_xva.shape} | "
      f"arquitetura {N_IN}-{'-'.join(str(_h) for _h in HIDDEN)}-{N_OUT} "
      f"({_n_params:,} params) | dtype {_DTYPE.__name__}")

_gpus = []
_TRAIN_GPU = False
try:
    import tensorflow as _tf
    _gpus = _tf.config.list_physical_devices("GPU")
except Exception:
    _tf = None

if _tf is not None and len(_gpus) >= 1:
    # ---------------- GPU PATH (Kaggle 2x T4, MirroredStrategy) ----------------
    # GPU treina em float32 de proposito: a T4 roda FP64 a 1/32 da velocidade
    # e o ganho de perda e nulo (o residuo e estrutural, nao numerico).
    if _DTYPE == np.float64:
        print("[kaggle] ETS2AI_F64=1 ignorado no caminho GPU (T4: FP64 = 1/32 "
              "da velocidade); use o caminho numpy/CPU para float64")
    print(f"[kaggle] GPU detectada: {len(_gpus)}x {_gpus[0][1]} — treinando com "
          f"tf.distribute.MirroredStrategy (todas as GPUs)")
    _strategy = _tf.distribute.MirroredStrategy()
    with _strategy.scope():
        _model = _tf.keras.Sequential()
        _model.add(_tf.keras.Input(shape=(13,)))
        for _h in HIDDEN:
            _model.add(_tf.keras.layers.Dense(_h, activation="tanh"))
        _model.add(_tf.keras.layers.Dense(3, activation="linear"))
        _model.compile(_tf.keras.optimizers.Adam(learning_rate=2e-3), loss="mse")
    _model.fit(_xtr, _ytr, epochs=400, batch_size=512, verbose=2)
    _w = _model.get_weights()
    _layers = [(np.asarray(_w[2 * i], dtype=np.float32),
                np.asarray(_w[2 * i + 1], dtype=np.float32))
               for i in range(len(_w) // 2)]
    _TRAIN_GPU = True
else:
    # ---------------- CPU fallback (identical math) ----------------
    print(f"[kaggle] sem GPU — treinando em numpy (CPU, {_DTYPE.__name__})")
    _layers, _hist = train(_xtr, _ytr, epochs=400, x_val=_xva, y_val=_yva,
                           dtype=_DTYPE)

_loss_tr = mse(forward(_xtr, _layers), _ytr)
_loss_va = mse(forward(_xva, _layers), _yva)
print(f"[kaggle] MSE treino {_loss_tr:.5f} | validacao {_loss_va:.5f} (meta <= {LOSS_TARGET})")

_agg, _rows = closed_loop_eval(_layers, n_roads=5)
print(f"[kaggle] circuito fechado: {_agg['finish_rate']*100:.0f}% rotas, "
      f"{_agg['in_lane_pct']*100:.0f}% faixa, {_agg['avg_speed_kmh']:.0f} km/h, "
      f"radares {_agg['radar_compliance']*100:.0f}%, dock {_agg['dock_rate']*100:.0f}%")

save_weights(_os.path.join(_OUT, "model-weights.json"), _layers,
             model_meta(epochs=400, samples=int(len(_xtr) + len(_xva)), loss=_loss_va,
                        dtype="float64" if _DTYPE == np.float64 else None))
with open(_os.path.join(_OUT, "metrics.json"), "w", encoding="utf-8") as _f:
    _json.dump({"mse_train": _loss_tr, "mse_val": _loss_va,
                "loss_target": LOSS_TARGET, "closed_loop": _agg,
                "trained_on_gpu": _TRAIN_GPU, "n_gpus": len(_gpus),
                "dtype": _DTYPE.__name__}, _f, indent=2)
print(f"[kaggle] saida: model-weights.json + metrics.json (gpu={_TRAIN_GPU}, "
      f"ngpus={len(_gpus)}, dtype={_DTYPE.__name__})")

